# Setup and configuration

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
ROOT_DIR = Path.cwd()

STAGE1_OUTPUT_DIR = (
    ROOT_DIR
    / "outputs"
    / "stage1"
)

STAGE2_OUTPUT_DIR = (
    ROOT_DIR
    / "outputs"
    / "stage2"
)

STAGE2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [3]:
RANDOM_SEED = 42
MINIMUM_TRAINING_CAMPAIGNS = 10

MIN_SIGNAL_TRADES = 20
MIN_CONTROL_TRADES = 20

N_BOOTSTRAP_SAMPLES = 10_000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95

HIGH_SIGNAL_QUANTILES = (
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
)

LOW_SIGNAL_QUANTILES = (
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
)

STARTING_BALANCE = 5_000.0
PROFIT_TARGET_RATE = 0.08
MAX_DRAWDOWN_RATE = 0.04

PROFIT_TARGET_AMOUNT = (
    STARTING_BALANCE
    * PROFIT_TARGET_RATE
)

DRAWDOWN_LIMIT_AMOUNT = (
    STARTING_BALANCE
    * MAX_DRAWDOWN_RATE
)

REALIZED_DRAWDOWN_BOUNDARY = (
    -DRAWDOWN_LIMIT_AMOUNT
)

np.random.seed(RANDOM_SEED)

In [4]:
if not STAGE1_OUTPUT_DIR.exists():
    raise FileNotFoundError(
        "Stage 1 output directory was not found: "
        f"{STAGE1_OUTPUT_DIR}"
    )

print(
    "Stage 1 input directory:",
    STAGE1_OUTPUT_DIR,
)

print(
    "Stage 2 output directory:",
    STAGE2_OUTPUT_DIR,
)

Stage 1 input directory: c:\Desktop\C22-veNTUre\outputs\stage1
Stage 2 output directory: c:\Desktop\C22-veNTUre\outputs\stage2


## Common Walk-Forward Testing Functions

The functions below provide a consistent testing framework for all feature
families.

For continuous features:

1. Candidate thresholds are learned using earlier campaigns only.
2. The threshold producing the largest training uplift is selected.
3. That fixed threshold is evaluated on the next unseen campaign.
4. Campaign-level out-of-sample uplift is summarized with a bootstrap
   confidence interval.

Binary and categorical features use the same campaign walk-forward structure
without threshold learning.

### Campaign order

In [5]:
def get_chronological_campaign_order(
    data: pd.DataFrame,
    campaign_column: str = "campaign_id",
    time_column: str = "open_date_time",
) -> list:
    """Returns campaign IDs ordered by their earliest observed timestamp.

    Campaign walk-forward testing must follow chronological order rather than
    assuming that campaign identifiers are inherently time ordered.

    Args:
        data: Table containing campaign identifiers and timestamps.
        campaign_column: Column identifying each campaign.
        time_column: Timestamp column used to determine campaign chronology.

    Returns:
        Campaign identifiers ordered from earliest to latest campaign.
    """
    campaign_start_times = (
        data
        .groupby(
            campaign_column,
            as_index=False,
        )[time_column]
        .min()
        .sort_values(
            [
                time_column,
                campaign_column,
            ]
        )
    )

    return (
        campaign_start_times[
            campaign_column
        ]
        .tolist()
    )

### Learn quantile threshold

In [6]:
def learn_quantile_threshold(
    train_data: pd.DataFrame,
    feature_column: str,
    target_column: str,
    signal_direction: str,
    candidate_quantiles: tuple,
    min_signal_trades: int = MIN_SIGNAL_TRADES,
    min_control_trades: int = MIN_CONTROL_TRADES,
) -> dict | None:
    """Learns the best training-only quantile threshold for a feature.

    Each candidate quantile is converted into a threshold using only the
    training campaigns. Signal and control observations are then compared
    using mean target value. The candidate producing the largest training
    uplift is selected.

    Args:
        train_data: Historical observations available during training.
        feature_column: Continuous feature used to define the signal.
        target_column: Target used to evaluate signal performance.
        signal_direction: Direction defining the signal. Must be either
            "high" or "low".
        candidate_quantiles: Quantiles used to generate candidate thresholds.
        min_signal_trades: Minimum required signal observations.
        min_control_trades: Minimum required control observations.

    Returns:
        Dictionary containing the selected quantile, threshold, training
        sample counts, and training uplift. Returns None when no candidate
        satisfies the minimum sample requirements.

    Raises:
        ValueError: If signal_direction is not "high" or "low".
    """
    if signal_direction not in {
        "high",
        "low",
    }:
        raise ValueError(
            "signal_direction must be "
            "'high' or 'low'."
        )

    candidate_results = []

    for quantile in candidate_quantiles:
        threshold = (
            train_data[
                feature_column
            ]
            .quantile(quantile)
        )

        if pd.isna(threshold):
            continue

        if signal_direction == "high":
            signal_mask = (
                train_data[
                    feature_column
                ]
                >= threshold
            )
        else:
            signal_mask = (
                train_data[
                    feature_column
                ]
                <= threshold
            )

        control_mask = ~signal_mask

        signal_count = int(
            signal_mask.sum()
        )

        control_count = int(
            control_mask.sum()
        )

        if (
            signal_count
            < min_signal_trades
            or control_count
            < min_control_trades
        ):
            continue

        signal_mean = (
            train_data.loc[
                signal_mask,
                target_column,
            ]
            .mean()
        )

        control_mean = (
            train_data.loc[
                control_mask,
                target_column,
            ]
            .mean()
        )

        candidate_results.append(
            {
                "quantile": quantile,
                "threshold": threshold,
                "training_signal_count": (
                    signal_count
                ),
                "training_control_count": (
                    control_count
                ),
                "training_uplift": (
                    signal_mean
                    - control_mean
                ),
            }
        )

    if not candidate_results:
        return None

    return max(
        candidate_results,
        key=lambda result: (
            result[
                "training_uplift"
            ]
        ),
    )

### Generic continuous-feature walk-forward test

In [7]:
def walk_forward_quantile_feature(
    data: pd.DataFrame,
    campaign_order: list,
    feature_column: str,
    target_column: str = (
        "reverse_profit_per_lot"
    ),
    signal_direction: str = "high",
    candidate_quantiles: tuple = (
        HIGH_SIGNAL_QUANTILES
    ),
    minimum_training_campaigns: int = (
        MINIMUM_TRAINING_CAMPAIGNS
    ),
    min_signal_trades: int = (
        MIN_SIGNAL_TRADES
    ),
    min_control_trades: int = (
        MIN_CONTROL_TRADES
    ),
) -> pd.DataFrame:
    """Walk-forward tests a continuous feature by campaign.

    For every test campaign, the signal threshold is learned exclusively
    from earlier campaigns. The learned threshold is then applied unchanged
    to the unseen campaign. Performance is measured as the difference in
    mean target value between signal and control trades.

    Args:
        data: Eligible observations containing campaign, feature, and target.
        campaign_order: Campaign identifiers ordered chronologically.
        feature_column: Continuous feature being evaluated.
        target_column: Target used to evaluate fade performance.
        signal_direction: Whether high or low feature values define signal.
        candidate_quantiles: Training quantiles considered as thresholds.
        minimum_training_campaigns: Number of campaigns required before the
            first out-of-sample evaluation.
        min_signal_trades: Minimum training observations in the signal group.
        min_control_trades: Minimum training observations in the control
            group.

    Returns:
        DataFrame containing one row per out-of-sample campaign with the
        learned threshold and signal-versus-control performance.
    """
    fold_results = []

    available_campaigns = set(
        data["campaign_id"].unique()
    )

    ordered_campaigns = [
        campaign_id
        for campaign_id in campaign_order
        if campaign_id in available_campaigns
    ]

    for test_index in range(
        minimum_training_campaigns,
        len(ordered_campaigns),
    ):
        training_campaigns = (
            ordered_campaigns[
                :test_index
            ]
        )

        test_campaign = (
            ordered_campaigns[
                test_index
            ]
        )

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = (
            learn_quantile_threshold(
                train_data=train_data,
                feature_column=(
                    feature_column
                ),
                target_column=(
                    target_column
                ),
                signal_direction=(
                    signal_direction
                ),
                candidate_quantiles=(
                    candidate_quantiles
                ),
                min_signal_trades=(
                    min_signal_trades
                ),
                min_control_trades=(
                    min_control_trades
                ),
            )
        )

        if threshold_result is None:
            continue

        threshold = (
            threshold_result[
                "threshold"
            ]
        )

        if signal_direction == "high":
            signal_mask = (
                test_data[
                    feature_column
                ]
                >= threshold
            )
        else:
            signal_mask = (
                test_data[
                    feature_column
                ]
                <= threshold
            )

        signal_data = (
            test_data.loc[
                signal_mask
            ]
        )

        control_data = (
            test_data.loc[
                ~signal_mask
            ]
        )

        signal_mean = (
            signal_data[
                target_column
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                target_column
            ].mean()
            if not control_data.empty
            else np.nan
        )

        if (
            pd.notna(signal_mean)
            and pd.notna(control_mean)
        ):
            test_uplift = (
                signal_mean
                - control_mean
            )
        else:
            test_uplift = np.nan

        fold_results.append(
            {
                "test_campaign": (
                    test_campaign
                ),
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": (
                    threshold
                ),
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_signal_mean": (
                    signal_mean
                ),
                "test_control_mean": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    test_uplift
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

### Binary-feature walk-forward test

In [8]:
def walk_forward_binary_feature(
    data: pd.DataFrame,
    campaign_order: list,
    signal_column: str,
    target_column: str = (
        "reverse_profit_per_lot"
    ),
    signal_value=True,
    minimum_training_campaigns: int = (
        MINIMUM_TRAINING_CAMPAIGNS
    ),
) -> pd.DataFrame:
    """Walk-forward tests a binary behavioural feature by campaign.

    No threshold is learned because the signal definition is fixed in
    advance. Earlier campaigns are retained to preserve the same
    walk-forward evaluation horizon used by continuous features.

    Args:
        data: Eligible observations containing campaign, signal, and target.
        campaign_order: Campaign identifiers ordered chronologically.
        signal_column: Binary feature defining signal and control groups.
        target_column: Target used to evaluate fade performance.
        signal_value: Value identifying the signal group.
        minimum_training_campaigns: Number of earlier campaigns required
            before the first out-of-sample evaluation.

    Returns:
        DataFrame containing campaign-level signal-versus-control results.
    """
    fold_results = []

    available_campaigns = set(
        data["campaign_id"].unique()
    )

    ordered_campaigns = [
        campaign_id
        for campaign_id in campaign_order
        if campaign_id in available_campaigns
    ]

    for test_index in range(
        minimum_training_campaigns,
        len(ordered_campaigns),
    ):
        test_campaign = (
            ordered_campaigns[
                test_index
            ]
        )

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        signal_mask = (
            test_data[
                signal_column
            ]
            == signal_value
        )

        signal_data = (
            test_data.loc[
                signal_mask
            ]
        )

        control_data = (
            test_data.loc[
                ~signal_mask
            ]
        )

        signal_mean = (
            signal_data[
                target_column
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                target_column
            ].mean()
            if not control_data.empty
            else np.nan
        )

        if (
            pd.notna(signal_mean)
            and pd.notna(control_mean)
        ):
            test_uplift = (
                signal_mean
                - control_mean
            )
        else:
            test_uplift = np.nan

        fold_results.append(
            {
                "test_campaign": (
                    test_campaign
                ),
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_signal_mean": (
                    signal_mean
                ),
                "test_control_mean": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    test_uplift
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

### Categorical walk-forward test

In [9]:
def walk_forward_categorical_feature(
    data: pd.DataFrame,
    campaign_order: list,
    feature_column: str,
    signal_value,
    reference_value,
    target_column: str = (
        "reverse_profit_per_lot"
    ),
    minimum_training_campaigns: int = (
        MINIMUM_TRAINING_CAMPAIGNS
    ),
) -> pd.DataFrame:
    """Walk-forward compares two predefined categories by campaign.

    The signal category is compared only with a specified reference
    category, allowing interpretable comparisons such as SL-only trades
    versus trades with both a stop loss and take profit.

    Args:
        data: Eligible observations containing campaign, category, and target.
        campaign_order: Campaign identifiers ordered chronologically.
        feature_column: Categorical feature being evaluated.
        signal_value: Category treated as the fade signal.
        reference_value: Category used as the comparison group.
        target_column: Target used to evaluate fade performance.
        minimum_training_campaigns: Number of earlier campaigns required
            before the first out-of-sample evaluation.

    Returns:
        DataFrame containing campaign-level signal-versus-reference results.
    """
    fold_results = []

    available_campaigns = set(
        data["campaign_id"].unique()
    )

    ordered_campaigns = [
        campaign_id
        for campaign_id in campaign_order
        if campaign_id in available_campaigns
    ]

    for test_index in range(
        minimum_training_campaigns,
        len(ordered_campaigns),
    ):
        test_campaign = (
            ordered_campaigns[
                test_index
            ]
        )

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        signal_data = test_data.loc[
            test_data[
                feature_column
            ].eq(
                signal_value
            )
        ]

        control_data = test_data.loc[
            test_data[
                feature_column
            ].eq(
                reference_value
            )
        ]

        signal_mean = (
            signal_data[
                target_column
            ].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[
                target_column
            ].mean()
            if not control_data.empty
            else np.nan
        )

        if (
            pd.notna(signal_mean)
            and pd.notna(control_mean)
        ):
            test_uplift = (
                signal_mean
                - control_mean
            )
        else:
            test_uplift = np.nan

        fold_results.append(
            {
                "test_campaign": (
                    test_campaign
                ),
                "signal_category": (
                    signal_value
                ),
                "reference_category": (
                    reference_value
                ),
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_signal_mean": (
                    signal_mean
                ),
                "test_control_mean": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    test_uplift
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

### Combined continuous + binary walk-forward test

In [10]:
def walk_forward_quantile_binary_combination(
    data: pd.DataFrame,
    campaign_order: list,
    feature_column: str,
    binary_column: str,
    binary_signal_value=True,
    target_column: str = "reverse_profit_per_lot",
    signal_direction: str = "low",
    candidate_quantiles: tuple = LOW_SIGNAL_QUANTILES,
    minimum_training_campaigns: int = MINIMUM_TRAINING_CAMPAIGNS,
    min_signal_trades: int = MIN_SIGNAL_TRADES,
    min_control_trades: int = MIN_CONTROL_TRADES,
) -> pd.DataFrame:
    """Walk-forward tests a quantile condition combined with a binary signal.

    The continuous-feature threshold is learned using earlier campaigns only.
    The out-of-sample signal requires both the learned quantile condition and
    the specified binary condition to hold.

    Args:
        data: Eligible observations containing campaign, feature, binary
            condition, and target columns.
        campaign_order: Campaign identifiers ordered chronologically.
        feature_column: Continuous feature used for threshold learning.
        binary_column: Binary feature combined with the threshold condition.
        binary_signal_value: Binary value required for the combined signal.
        target_column: Target used to evaluate fade performance.
        signal_direction: Whether high or low continuous values define signal.
        candidate_quantiles: Training-only quantiles considered as thresholds.
        minimum_training_campaigns: Number of earlier campaigns required
            before the first out-of-sample test.
        min_signal_trades: Minimum training observations in the quantile signal.
        min_control_trades: Minimum training observations in the quantile
            control group.

    Returns:
        DataFrame containing one row per out-of-sample campaign.
    """
    fold_results = []

    available_campaigns = set(
        data["campaign_id"].unique()
    )

    ordered_campaigns = [
        campaign_id
        for campaign_id in campaign_order
        if campaign_id in available_campaigns
    ]

    for test_index in range(
        minimum_training_campaigns,
        len(ordered_campaigns),
    ):
        training_campaigns = ordered_campaigns[:test_index]
        test_campaign = ordered_campaigns[test_index]

        train_data = data.loc[
            data["campaign_id"].isin(
                training_campaigns
            )
        ]

        test_data = data.loc[
            data["campaign_id"].eq(
                test_campaign
            )
        ].copy()

        threshold_result = learn_quantile_threshold(
            train_data=train_data,
            feature_column=feature_column,
            target_column=target_column,
            signal_direction=signal_direction,
            candidate_quantiles=candidate_quantiles,
            min_signal_trades=min_signal_trades,
            min_control_trades=min_control_trades,
        )

        if threshold_result is None:
            continue

        threshold = threshold_result["threshold"]

        if signal_direction == "high":
            quantile_signal = (
                test_data[feature_column]
                >= threshold
            )
        else:
            quantile_signal = (
                test_data[feature_column]
                <= threshold
            )

        combined_signal = (
            quantile_signal
            & test_data[
                binary_column
            ].eq(
                binary_signal_value
            )
        )

        signal_data = test_data.loc[
            combined_signal
        ]

        control_data = test_data.loc[
            ~combined_signal
        ]

        signal_mean = (
            signal_data[target_column].mean()
            if not signal_data.empty
            else np.nan
        )

        control_mean = (
            control_data[target_column].mean()
            if not control_data.empty
            else np.nan
        )

        test_uplift = (
            signal_mean - control_mean
            if (
                pd.notna(signal_mean)
                and pd.notna(control_mean)
            )
            else np.nan
        )

        fold_results.append(
            {
                "test_campaign": test_campaign,
                "learned_quantile": (
                    threshold_result[
                        "quantile"
                    ]
                ),
                "learned_threshold": threshold,
                "test_signal_trades": (
                    len(signal_data)
                ),
                "test_control_trades": (
                    len(control_data)
                ),
                "test_signal_mean": (
                    signal_mean
                ),
                "test_control_mean": (
                    control_mean
                ),
                "test_uplift_vs_control": (
                    test_uplift
                ),
            }
        )

    return pd.DataFrame(
        fold_results
    )

### Campaign bootstrap confidence interval

In [11]:
def bootstrap_campaign_uplift_ci(
    results: pd.DataFrame,
    uplift_column: str = (
        "test_uplift_vs_control"
    ),
    n_bootstrap: int = (
        N_BOOTSTRAP_SAMPLES
    ),
    confidence_level: float = (
        BOOTSTRAP_CONFIDENCE_LEVEL
    ),
    random_state: int = RANDOM_SEED,
) -> tuple[float, float]:
    """Calculates a bootstrap confidence interval for mean OOS uplift.

    Campaigns, rather than individual trades, are resampled so that the
    uncertainty estimate reflects variation in out-of-sample performance
    across campaigns.

    Args:
        results: Walk-forward results containing one row per test campaign.
        uplift_column: Column containing campaign-level OOS uplift.
        n_bootstrap: Number of bootstrap resamples.
        confidence_level: Confidence level for the returned interval.
        random_state: Random seed used for reproducibility.

    Returns:
        Lower and upper bounds of the bootstrap confidence interval.
    """
    campaign_uplifts = (
        results[
            uplift_column
        ]
        .dropna()
        .to_numpy()
    )

    if len(campaign_uplifts) == 0:
        return (
            np.nan,
            np.nan,
        )

    rng = np.random.default_rng(
        random_state
    )

    bootstrap_means = np.empty(
        n_bootstrap
    )

    for bootstrap_index in range(
        n_bootstrap
    ):
        bootstrap_sample = rng.choice(
            campaign_uplifts,
            size=len(
                campaign_uplifts
            ),
            replace=True,
        )

        bootstrap_means[
            bootstrap_index
        ] = bootstrap_sample.mean()

    alpha = (
        1.0
        - confidence_level
    )

    lower_bound = np.quantile(
        bootstrap_means,
        alpha / 2.0,
    )

    upper_bound = np.quantile(
        bootstrap_means,
        1.0 - alpha / 2.0,
    )

    return (
        float(lower_bound),
        float(upper_bound),
    )

### Robustness clarification

In [12]:
def classify_robustness(
    ci_lower: float,
    ci_upper: float,
) -> str:
    """Classifies feature robustness from its confidence interval.

    Args:
        ci_lower: Lower bound of the confidence interval.
        ci_upper: Upper bound of the confidence interval.

    Returns:
        "robust_positive" when the complete interval is above zero,
        "robust_negative" when the complete interval is below zero,
        otherwise "not_robust".
    """
    if (
        pd.isna(ci_lower)
        or pd.isna(ci_upper)
    ):
        return "insufficient_data"

    if ci_lower > 0:
        return "robust_positive"

    if ci_upper < 0:
        return "robust_negative"

    return "not_robust"

### Standard walk-forward summary

In [13]:
def summarize_walk_forward(
    results: pd.DataFrame,
    feature_family: str,
    feature_name: str,
    feature_column: str,
    signal_description: str,
) -> dict:
    """Summarizes one walk-forward feature check in a standard format.

    Args:
        results: Campaign-level walk-forward results.
        feature_family: Behavioural family containing the feature.
        feature_name: Human-readable name of the tested signal.
        feature_column: Exact engineered feature column used in the test.
        signal_description: Plain-language definition of the tested signal.

    Returns:
        Dictionary containing OOS performance, confidence interval,
        robustness classification, and identifying metadata.
    """
    valid_uplifts = (
        results[
            "test_uplift_vs_control"
        ]
        .dropna()
    )

    if valid_uplifts.empty:
        return {
            "feature_family": (
                feature_family
            ),
            "feature_name": (
                feature_name
            ),
            "feature_column": (
                feature_column
            ),
            "signal_description": (
                signal_description
            ),
            "test_campaigns": 0,
            "mean_oos_uplift": np.nan,
            "median_oos_uplift": np.nan,
            "positive_campaign_share": (
                np.nan
            ),
            "ci_lower": np.nan,
            "ci_upper": np.nan,
            "robustness": (
                "insufficient_data"
            ),
        }

    ci_lower, ci_upper = (
        bootstrap_campaign_uplift_ci(
            results
        )
    )

    summary = {
        "feature_family": (
            feature_family
        ),
        "feature_name": (
            feature_name
        ),
        "feature_column": (
            feature_column
        ),
        "signal_description": (
            signal_description
        ),
        "test_campaigns": (
            len(valid_uplifts)
        ),
        "mean_oos_uplift": (
            valid_uplifts.mean()
        ),
        "median_oos_uplift": (
            valid_uplifts.median()
        ),
        "positive_campaign_share": (
            valid_uplifts.gt(0).mean()
        ),
        "ci_lower": (
            ci_lower
        ),
        "ci_upper": (
            ci_upper
        ),
        "robustness": (
            classify_robustness(
                ci_lower,
                ci_upper,
            )
        ),
    }

    return summary

## Output record containers

In [14]:
feature_check_records = []
feature_dictionary_records = []

# Load stage 1 outputs

## Define stage 1 files

In [15]:
stage1_file_paths = {
    "trades": (
        STAGE1_OUTPUT_DIR
        / "trades.parquet"
    ),
    "trades_with_ideas": (
        STAGE1_OUTPUT_DIR
        / "trades_with_ideas.parquet"
    ),
    "trades_with_previous_completed": (
        STAGE1_OUTPUT_DIR
        / "trades_with_previous_completed.parquet"
    ),
    "idea_features": (
        STAGE1_OUTPUT_DIR
        / "idea_features_stage1.parquet"
    ),
}

## Load tables

In [16]:
trades = pd.read_parquet(
    stage1_file_paths[
        "trades"
    ]
)

trades_with_ideas = pd.read_parquet(
    stage1_file_paths[
        "trades_with_ideas"
    ]
)

trades_with_previous_completed = (
    pd.read_parquet(
        stage1_file_paths[
            "trades_with_previous_completed"
        ]
    )
)

idea_features = pd.read_parquet(
    stage1_file_paths[
        "idea_features"
    ]
)

## Establish campaign chronology

In [17]:
campaign_order = (
    get_chronological_campaign_order(
        data=trades,
        campaign_column=(
            "campaign_id"
        ),
        time_column=(
            "open_date_time"
        ),
    )
)

# Create Trade-Level Feature Table

The trade-level feature table contains one row per trade.

It begins with the Stage 1 trade-and-idea mapping and attaches only information
from previously completed trades. Additional Stage 2 features are added to
this table within their respective feature-family sections.

## Initialize trade-level table

In [18]:
trade_features_table = (
    trades_with_ideas
    .copy()
)

## Attach previous completed trade information

In [19]:
previous_completed_columns = [
    "trade_row_id",
    "previous_completed_close_date_time",
    "previous_completed_net_profit",
    "previous_completed_amount",
    "previous_completed_position_id",
    "previous_completed_was_loss",
    "previous_completed_was_win",
    "reentry_gap_minutes",
]

trade_features_table = (
    trade_features_table
    .merge(
        trades_with_previous_completed[
            previous_completed_columns
        ],
        on="trade_row_id",
        how="left",
        validate="one_to_one",
    )
)

## Previous-to-current position-size ratio

In [20]:
trade_features_table[
    "current_to_previous_amount_ratio"
] = (
    trade_features_table[
        "amount"
    ]
    / trade_features_table[
        "previous_completed_amount"
    ]
)

trade_features_table[
    "current_to_previous_amount_ratio"
] = (
    trade_features_table[
        "current_to_previous_amount_ratio"
    ]
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)

## Standard trade ordering

In [21]:
trade_features_table = (
    trade_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
            "trade_row_id",
        ]
    )
    .reset_index(drop=True)
)

# Create Idea-Level Feature Table

The idea-level feature table contains one row per trade idea and provides the
base for idea-level historical behavioural features.

In [22]:
idea_features_table = (
    idea_features
    .copy()
)

idea_features_table = (
    idea_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "idea_start_time",
            "idea_end_time",
            "idea_id",
        ]
    )
    .reset_index(drop=True)
)

# Create reverseProfit-per-lot target

In [23]:
trades[
    "reverse_profit_per_lot"
] = (
    trades[
        "reverse_profit"
    ]
    / trades[
        "amount"
    ]
)

In [24]:
trade_features_table = (
    trade_features_table
    .merge(
        trades[
            [
                "trade_row_id",
                "reverse_profit_per_lot",
            ]
        ],
        on="trade_row_id",
        how="left",
        validate="one_to_one",
    )
)

# Loss response

## Hypothesis

Trades entered immediately after a completed loss may reflect reactive
decision-making. Two dimensions are measured:

1. how quickly the trader re-enters after a loss; and
2. whether the trader increases position size after that loss.

All loss-response features use only the most recent trade that had already
completed when the current trade opened.

## Build features

### Post-loss indicator

In [25]:
trade_features_table[
    "post_loss_entry"
] = (
    trade_features_table[
        "previous_completed_was_loss"
    ]
    .fillna(False)
    .astype(bool)
)

### Post-loss re-entry gap

In [26]:
trade_features_table[
    "post_loss_reentry_gap_minutes"
] = (
    trade_features_table[
        "reentry_gap_minutes"
    ]
    .where(
        trade_features_table[
            "post_loss_entry"
        ]
    )
)

### Post-loss position-size ratio

In [27]:
trade_features_table[
    "post_loss_amount_ratio"
] = (
    trade_features_table[
        "current_to_previous_amount_ratio"
    ]
    .where(
        trade_features_table[
            "post_loss_entry"
        ]
    )
)

In [28]:
trade_features_table[
    "post_loss_amount_increased"
] = (
    trade_features_table[
        "post_loss_entry"
    ]
    & (
        trade_features_table[
            "post_loss_amount_ratio"
        ]
        > 1
    )
)

In [29]:
trade_features_table[
    "post_loss_features_available"
] = (
    trade_features_table[
        "post_loss_entry"
    ]
    & trade_features_table[
        "post_loss_reentry_gap_minutes"
    ].notna()
    & trade_features_table[
        "post_loss_amount_ratio"
    ].notna()
)

## Data dictionary

In [30]:
feature_dictionary_records.extend(
    [
        {
            "feature_family": "Loss Response",
            "feature_name": "post_loss_entry",
            "level": "trade",
            "meaning": (
                "Indicates that the most recent completed trade "
                "before the current entry was a loss."
            ),
            "behaviour": (
                "Identifies trades occurring immediately after "
                "a realized loss."
            ),
            "calculation": (
                "True when previous_completed_was_loss is True."
            ),
            "pre_trade_only": True,
            "tested": False,
        },
        {
            "feature_family": "Loss Response",
            "feature_name": "post_loss_reentry_gap_minutes",
            "level": "trade",
            "meaning": (
                "Minutes between the previous completed losing "
                "trade and the current trade entry."
            ),
            "behaviour": (
                "Measures how quickly the trader re-enters "
                "after a loss."
            ),
            "calculation": (
                "Current open time minus the close time of the "
                "most recent completed losing trade."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": "Loss Response",
            "feature_name": "post_loss_amount_ratio",
            "level": "trade",
            "meaning": (
                "Current trade amount relative to the amount of "
                "the previous completed losing trade."
            ),
            "behaviour": (
                "Measures position-size reaction after a loss."
            ),
            "calculation": (
                "Current amount divided by previous completed "
                "trade amount, restricted to post-loss entries."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": "Loss Response",
            "feature_name": "post_loss_amount_increased",
            "level": "trade",
            "meaning": (
                "Indicates that position size increased after "
                "the previous completed loss."
            ),
            "behaviour": (
                "Captures post-loss size escalation."
            ),
            "calculation": (
                "post_loss_amount_ratio > 1 for a post-loss entry."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
    ]
)

## Test features

In [31]:
post_loss_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "post_loss_features_available"
        ]
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "post_loss_reentry_gap_minutes",
            "post_loss_amount_ratio",
            "post_loss_amount_increased",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

### Fast post-loss re-entry

In [32]:
fast_reentry_results = (
    walk_forward_quantile_feature(
        data=post_loss_test_data,
        campaign_order=campaign_order,
        feature_column=(
            "post_loss_reentry_gap_minutes"
        ),
        signal_direction="low",
        candidate_quantiles=(
            LOW_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=fast_reentry_results,
        feature_family="Loss Response",
        feature_name=(
            "Fast post-loss re-entry"
        ),
        feature_column=(
            "post_loss_reentry_gap_minutes"
        ),
        signal_description=(
            "Post-loss re-entry gap is unusually short "
            "relative to thresholds learned from earlier campaigns."
        ),
    )
)

### Post-lost size escalation

In [33]:
post_loss_size_results = (
    walk_forward_binary_feature(
        data=post_loss_test_data,
        campaign_order=campaign_order,
        signal_column=(
            "post_loss_amount_increased"
        ),
        signal_value=True,
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=post_loss_size_results,
        feature_family="Loss Response",
        feature_name=(
            "Post-loss size escalation"
        ),
        feature_column=(
            "post_loss_amount_increased"
        ),
        signal_description=(
            "Current position size is larger than the previous "
            "completed losing trade."
        ),
    )
)

### Fast re-entry + size escalation

In [34]:
post_loss_combination_results = (
    walk_forward_quantile_binary_combination(
        data=post_loss_test_data,
        campaign_order=campaign_order,
        feature_column=(
            "post_loss_reentry_gap_minutes"
        ),
        binary_column=(
            "post_loss_amount_increased"
        ),
        binary_signal_value=True,
        signal_direction="low",
        candidate_quantiles=(
            LOW_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=post_loss_combination_results,
        feature_family="Loss Response",
        feature_name=(
            "Fast re-entry with size escalation"
        ),
        feature_column=(
            "post_loss_reentry_gap_minutes "
            "+ post_loss_amount_increased"
        ),
        signal_description=(
            "Trade both re-enters unusually quickly after a loss "
            "and increases position size."
        ),
    )
)

# Trading Activity and Timing

## Hypothesis

Changes in trading intensity or entry pace may indicate increasingly reactive
behaviour. The family measures cumulative trading intensity, recent bursts of
activity, and acceleration relative to the trader's own historical entry pace.

## Build features

### Cumulative event-count helper

In [35]:
def attach_past_event_count(
    target_table: pd.DataFrame,
    event_table: pd.DataFrame,
    group_columns: list[str],
    target_time_column: str,
    event_time_column: str,
    output_column: str,
) -> pd.DataFrame:
    """Attaches the number of strictly earlier events to each target row.

    Args:
        target_table: Table receiving the historical count.
        event_table: Table containing historical events.
        group_columns: Columns defining independent histories.
        target_time_column: Timestamp at which history is evaluated.
        event_time_column: Timestamp when each event occurred.
        output_column: Name assigned to the generated count.

    Returns:
        Copy of the target table containing the historical event count.
    """
    target = target_table.copy()

    event_counts = (
        event_table
        .groupby(
            group_columns
            + [event_time_column],
            dropna=False,
        )
        .size()
        .rename("_events_at_timestamp")
        .reset_index()
    )

    event_counts = event_counts.sort_values(
        group_columns
        + [event_time_column]
    )

    event_counts[output_column] = (
        event_counts
        .groupby(
            group_columns,
            dropna=False,
        )["_events_at_timestamp"]
        .cumsum()
    )

    output_groups = []

    for group_key, target_group in target.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=event_counts.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                event_mask &= (
                    event_counts[
                        column
                    ].isna()
                )
            else:
                event_mask &= (
                    event_counts[
                        column
                    ].eq(value)
                )

        group_events = (
            event_counts.loc[
                event_mask,
                [
                    event_time_column,
                    output_column,
                ],
            ]
            .sort_values(
                event_time_column
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if group_events.empty:
            current_group[
                output_column
            ] = 0

            output_groups.append(
                current_group
            )
            continue

        merge_time_column = (
            f"_past_{output_column}_time"
        )

        group_events = (
            group_events.rename(
                columns={
                    event_time_column: (
                        merge_time_column
                    )
                }
            )
        )

        current_group = pd.merge_asof(
            left=current_group,
            right=group_events,
            left_on=target_time_column,
            right_on=merge_time_column,
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            output_column
        ] = (
            current_group[
                output_column
            ]
            .fillna(0)
            .astype(int)
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

### Cumulative trading intensity

In [36]:
trade_features_table = (
    attach_past_event_count(
        target_table=(
            trade_features_table
        ),
        event_table=trades,
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column=(
            "open_date_time"
        ),
        event_time_column=(
            "open_date_time"
        ),
        output_column=(
            "past_trade_open_count"
        ),
    )
)

In [37]:
idea_start_events = (
    idea_features_table[
        [
            "account_id",
            "campaign_id",
            "idea_id",
            "idea_start_time",
        ]
    ]
    .drop_duplicates(
        subset="idea_id"
    )
    .copy()
)

In [38]:
trade_features_table = (
    attach_past_event_count(
        target_table=(
            trade_features_table
        ),
        event_table=(
            idea_start_events
        ),
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column=(
            "open_date_time"
        ),
        event_time_column=(
            "idea_start_time"
        ),
        output_column=(
            "past_idea_start_count"
        ),
    )
)

In [39]:
trade_features_table[
    "first_trade_open_date_time"
] = (
    trade_features_table
    .groupby(
        [
            "account_id",
            "campaign_id",
        ]
    )["open_date_time"]
    .transform("min")
)

trade_features_table[
    "elapsed_active_minutes"
] = (
    (
        trade_features_table[
            "open_date_time"
        ]
        - trade_features_table[
            "first_trade_open_date_time"
        ]
    )
    .dt.total_seconds()
    / 60
)

trade_features_table[
    "elapsed_active_hours"
] = (
    trade_features_table[
        "elapsed_active_minutes"
    ]
    / 60
)

valid_elapsed_time = (
    trade_features_table[
        "elapsed_active_hours"
    ].gt(0)
)

trade_features_table[
    "past_trades_opened_per_active_hour"
] = np.where(
    valid_elapsed_time,
    (
        trade_features_table[
            "past_trade_open_count"
        ]
        / trade_features_table[
            "elapsed_active_hours"
        ]
    ),
    np.nan,
)

trade_features_table[
    "past_ideas_started_per_active_hour"
] = np.where(
    valid_elapsed_time,
    (
        trade_features_table[
            "past_idea_start_count"
        ]
        / trade_features_table[
            "elapsed_active_hours"
        ]
    ),
    np.nan,
)

### Recent activity helper

In [40]:
def attach_rolling_event_counts(
    target_table: pd.DataFrame,
    event_table: pd.DataFrame,
    group_columns: list[str],
    target_time_column: str,
    event_time_column: str,
    window_minutes: list[int],
    feature_prefix: str,
) -> pd.DataFrame:
    """Attaches counts of strictly earlier events within rolling windows.

    An event is counted for target time t and window w when
    t - w <= event_time < t.

    Args:
        target_table: Table receiving rolling event-count features.
        event_table: Table containing historical events.
        group_columns: Columns defining independent histories.
        target_time_column: Timestamp when history is evaluated.
        event_time_column: Timestamp when each event occurred.
        window_minutes: Rolling-window lengths in minutes.
        feature_prefix: Prefix used for generated feature names.

    Returns:
        Copy of the target table containing rolling counts.
    """
    result_groups = []

    for group_key, target_group in target_table.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=event_table.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                event_mask &= (
                    event_table[
                        column
                    ].isna()
                )
            else:
                event_mask &= (
                    event_table[
                        column
                    ].eq(value)
                )

        group_events = (
            event_table.loc[
                event_mask,
                event_time_column,
            ]
            .dropna()
            .sort_values()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        target_times = (
            current_group[
                target_time_column
            ]
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        event_times = (
            group_events
            .to_numpy(
                dtype="datetime64[ns]"
            )
        )

        for window in window_minutes:
            left_boundaries = (
                target_times
                - np.timedelta64(
                    window,
                    "m",
                )
            )

            left_indices = np.searchsorted(
                event_times,
                left_boundaries,
                side="left",
            )

            right_indices = np.searchsorted(
                event_times,
                target_times,
                side="left",
            )

            feature_column = (
                f"{feature_prefix}"
                f"_past_{window}_minutes"
            )

            current_group[
                feature_column
            ] = (
                right_indices
                - left_indices
            )

        result_groups.append(
            current_group
        )

    return pd.concat(
        result_groups,
        ignore_index=True,
    )

In [41]:
trade_features_table = (
    attach_rolling_event_counts(
        target_table=(
            trade_features_table
        ),
        event_table=trades,
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column=(
            "open_date_time"
        ),
        event_time_column=(
            "open_date_time"
        ),
        window_minutes=[
            15,
            30,
            60,
        ],
        feature_prefix=(
            "trades_opened"
        ),
    )
)

In [42]:
trade_features_table = (
    attach_rolling_event_counts(
        target_table=(
            trade_features_table
        ),
        event_table=(
            idea_start_events
        ),
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column=(
            "open_date_time"
        ),
        event_time_column=(
            "idea_start_time"
        ),
        window_minutes=[
            15,
            30,
            60,
        ],
        feature_prefix=(
            "ideas_started"
        ),
    )
)

### Relative trading pace helper

In [43]:
def attach_entry_spacing_features(
    target_table: pd.DataFrame,
    event_table: pd.DataFrame,
    group_columns: list[str],
    target_time_column: str,
    event_time_column: str,
    tie_breaker_columns: list[str],
    previous_gap_column: str,
    past_median_gap_column: str,
    gap_ratio_column: str,
) -> pd.DataFrame:
    """Attaches pre-entry spacing and historical pace features.

    Only events occurring strictly before the target timestamp are used.
    The current entry gap is compared with the historical median event gap.

    Args:
        target_table: Table receiving entry-spacing features.
        event_table: Table containing historical entry events.
        group_columns: Columns defining independent histories.
        target_time_column: Timestamp at which spacing is measured.
        event_time_column: Historical event timestamp.
        tie_breaker_columns: Columns providing deterministic event ordering.
        previous_gap_column: Output column for current entry gap in minutes.
        past_median_gap_column: Output column for historical median gap.
        gap_ratio_column: Output column for current gap divided by the
            historical median gap.

    Returns:
        Copy of the target table containing entry-spacing features.
    """
    target = target_table.copy()
    events = event_table.copy()

    output_groups = []

    for group_key, target_group in target.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=events.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                event_mask &= (
                    events[
                        column
                    ].isna()
                )
            else:
                event_mask &= (
                    events[
                        column
                    ].eq(value)
                )

        group_events = (
            events.loc[
                event_mask
            ]
            .sort_values(
                [
                    event_time_column
                ]
                + tie_breaker_columns
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                target_time_column
            )
            .copy()
        )

        if group_events.empty:
            current_group[
                previous_gap_column
            ] = np.nan

            current_group[
                past_median_gap_column
            ] = np.nan

            current_group[
                gap_ratio_column
            ] = np.nan

            output_groups.append(
                current_group
            )
            continue

        event_gap_column = (
            "_entry_gap_minutes"
        )

        group_events[
            event_gap_column
        ] = (
            group_events[
                event_time_column
            ]
            .diff()
            .dt.total_seconds()
            / 60
        )

        group_events[
            past_median_gap_column
        ] = (
            group_events[
                event_gap_column
            ]
            .expanding(
                min_periods=1
            )
            .median()
        )

        merge_time_column = (
            "_previous_event_time"
        )

        events_for_merge = (
            group_events[
                [
                    event_time_column,
                    past_median_gap_column,
                ]
            ]
            .rename(
                columns={
                    event_time_column: (
                        merge_time_column
                    )
                }
            )
        )

        current_group = pd.merge_asof(
            left=current_group,
            right=events_for_merge,
            left_on=(
                target_time_column
            ),
            right_on=(
                merge_time_column
            ),
            direction="backward",
            allow_exact_matches=False,
        )

        current_group[
            previous_gap_column
        ] = (
            (
                current_group[
                    target_time_column
                ]
                - current_group[
                    merge_time_column
                ]
            )
            .dt.total_seconds()
            / 60
        )

        valid_ratio = (
            current_group[
                previous_gap_column
            ].ge(0)
            & current_group[
                past_median_gap_column
            ].gt(0)
        )

        current_group[
            gap_ratio_column
        ] = np.where(
            valid_ratio,
            (
                current_group[
                    previous_gap_column
                ]
                / current_group[
                    past_median_gap_column
                ]
            ),
            np.nan,
        )

        current_group = (
            current_group.drop(
                columns=[
                    merge_time_column
                ],
                errors="ignore",
            )
        )

        output_groups.append(
            current_group
        )

    return pd.concat(
        output_groups,
        ignore_index=True,
    )

In [44]:
trade_features_table = (
    attach_entry_spacing_features(
        target_table=(
            trade_features_table
        ),
        event_table=trades,
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column=(
            "open_date_time"
        ),
        event_time_column=(
            "open_date_time"
        ),
        tie_breaker_columns=[
            "trade_row_id",
        ],
        previous_gap_column=(
            "minutes_since_previous_trade_open"
        ),
        past_median_gap_column=(
            "past_median_trade_open_gap_minutes"
        ),
        gap_ratio_column=(
            "trade_open_gap_to_past_median_ratio"
        ),
    )
)

In [45]:
idea_features_table = (
    attach_entry_spacing_features(
        target_table=(
            idea_features_table
        ),
        event_table=(
            idea_start_events
        ),
        group_columns=[
            "account_id",
            "campaign_id",
        ],
        target_time_column=(
            "idea_start_time"
        ),
        event_time_column=(
            "idea_start_time"
        ),
        tie_breaker_columns=[
            "idea_id",
        ],
        previous_gap_column=(
            "minutes_since_previous_idea_start"
        ),
        past_median_gap_column=(
            "past_median_idea_start_gap_minutes"
        ),
        gap_ratio_column=(
            "idea_start_gap_to_past_median_ratio"
        ),
    )
)

## Data dictionary

In [46]:
feature_dictionary_records.extend(
    [
        {
            "feature_family": "Trading Activity and Timing",
            "feature_name": "past_trades_opened_per_active_hour",
            "level": "trade",
            "meaning": "Historical trade-entry rate before the current trade.",
            "behaviour": "Cumulative trading intensity.",
            "calculation": (
                "Number of strictly earlier trade opens divided by "
                "elapsed hours since the account-campaign's first entry."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": "Trading Activity and Timing",
            "feature_name": "trades_opened_past_30_minutes",
            "level": "trade",
            "meaning": (
                "Number of trades opened during the 30 minutes "
                "immediately before the current entry."
            ),
            "behaviour": "Recent burst trading intensity.",
            "calculation": (
                "Count of trade opens where current time - 30 minutes "
                "<= historical open time < current open time."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": "Trading Activity and Timing",
            "feature_name": "trade_open_gap_to_past_median_ratio",
            "level": "trade",
            "meaning": (
                "Current entry gap relative to the trader's historical "
                "median entry gap."
            ),
            "behaviour": "Trading acceleration relative to normal pace.",
            "calculation": (
                "Minutes since previous trade open divided by the "
                "historical median trade-open gap."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
    ]
)

## Test features

### Cumulative intensity

In [47]:
cumulative_intensity_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_trades_opened_per_active_hour"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_trades_opened_per_active_hour",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

cumulative_intensity_results = (
    walk_forward_quantile_feature(
        data=(
            cumulative_intensity_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "past_trades_opened_per_active_hour"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=(
            cumulative_intensity_results
        ),
        feature_family=(
            "Trading Activity and Timing"
        ),
        feature_name=(
            "High cumulative trading intensity"
        ),
        feature_column=(
            "past_trades_opened_per_active_hour"
        ),
        signal_description=(
            "Historical trade-entry rate is unusually high."
        ),
    )
)

### Recent 30-minute intensity

In [48]:
recent_intensity_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "trades_opened_past_30_minutes"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "trades_opened_past_30_minutes",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

recent_intensity_results = (
    walk_forward_quantile_feature(
        data=(
            recent_intensity_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "trades_opened_past_30_minutes"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=(
            recent_intensity_results
        ),
        feature_family=(
            "Trading Activity and Timing"
        ),
        feature_name=(
            "High recent trading intensity"
        ),
        feature_column=(
            "trades_opened_past_30_minutes"
        ),
        signal_description=(
            "Unusually many trades were opened during the "
            "30 minutes before the current entry."
        ),
    )
)

### Fast relative trading pace

In [49]:
trading_pace_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "trade_open_gap_to_past_median_ratio"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "trade_open_gap_to_past_median_ratio",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

trading_pace_results = (
    walk_forward_quantile_feature(
        data=(
            trading_pace_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "trade_open_gap_to_past_median_ratio"
        ),
        signal_direction="low",
        candidate_quantiles=(
            LOW_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=(
            trading_pace_results
        ),
        feature_family=(
            "Trading Activity and Timing"
        ),
        feature_name=(
            "Unusually fast trading pace"
        ),
        feature_column=(
            "trade_open_gap_to_past_median_ratio"
        ),
        signal_description=(
            "Current trade is opened much sooner than the trader's "
            "own historical entry pace."
        ),
    )
)

# Challenge State and Drawdown Pressure

## Hypothesis

Trading behaviour may deteriorate as the account moves into drawdown or
approaches the challenge profit target. The features reconstruct the realized
account state using only trades completed before the current entry.

## Build features

### Realized P&L events

In [55]:
realized_pnl_events = (
    trades[
        [
            "account_id",
            "campaign_id",
            "close_date_time",
            "net_profit",
        ]
    ]
    .dropna(
        subset=[
            "close_date_time"
        ]
    )
    .groupby(
        [
            "account_id",
            "campaign_id",
            "close_date_time",
        ],
        as_index=False,
    )["net_profit"]
    .sum()
    .rename(
        columns={
            "net_profit":
                "realized_pnl_at_close"
        }
    )
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "close_date_time",
        ]
    )
    .reset_index(drop=True)
)

realized_pnl_events[
    "cumulative_realized_pnl"
] = (
    realized_pnl_events
    .groupby(
        [
            "campaign_id",
            "account_id",
        ]
    )[
        "realized_pnl_at_close"
    ]
    .cumsum()
)

### Attach realized challenge state

In [56]:
def attach_realized_challenge_state(
    target_table: pd.DataFrame,
    realized_events: pd.DataFrame,
) -> pd.DataFrame:
    """Attaches cumulative realized P&L known when each trade opens.

    The latest realized-P&L event whose close timestamp is less than or equal
    to the current trade's opening timestamp is used.

    Args:
        target_table: One-row-per-trade feature table.
        realized_events: Table of cumulative realized P&L close events.

    Returns:
        Copy of the target table containing pre-trade realized account state.
    """
    result_frames = []

    group_columns = [
        "campaign_id",
        "account_id",
    ]

    for group_key, target_group in target_table.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        event_mask = pd.Series(
            True,
            index=realized_events.index,
        )

        for column, value in zip(
            group_columns,
            group_key,
        ):
            if pd.isna(value):
                event_mask &= (
                    realized_events[
                        column
                    ].isna()
                )
            else:
                event_mask &= (
                    realized_events[
                        column
                    ].eq(value)
                )

        group_events = (
            realized_events.loc[
                event_mask,
                [
                    "close_date_time",
                    "cumulative_realized_pnl",
                ],
            ]
            .sort_values(
                "close_date_time"
            )
            .copy()
        )

        current_group = (
            target_group
            .sort_values(
                "open_date_time"
            )
            .copy()
        )

        if group_events.empty:
            current_group[
                "realized_pnl_before_trade"
            ] = 0.0

            current_group[
                "latest_realized_close_before_trade"
            ] = pd.NaT

            result_frames.append(
                current_group
            )
            continue

        group_events = (
            group_events.rename(
                columns={
                    "close_date_time": (
                        "latest_realized_close_before_trade"
                    )
                }
            )
        )

        current_group = pd.merge_asof(
            left=current_group,
            right=group_events,
            left_on="open_date_time",
            right_on=(
                "latest_realized_close_before_trade"
            ),
            direction="backward",
            allow_exact_matches=True,
        )

        current_group[
            "realized_pnl_before_trade"
        ] = (
            current_group[
                "cumulative_realized_pnl"
            ]
            .fillna(0.0)
        )

        current_group = (
            current_group.drop(
                columns=[
                    "cumulative_realized_pnl"
                ],
                errors="ignore",
            )
        )

        result_frames.append(
            current_group
        )

    return pd.concat(
        result_frames,
        ignore_index=True,
    )

In [57]:
trade_features_table = (
    attach_realized_challenge_state(
        target_table=(
            trade_features_table
        ),
        realized_events=(
            realized_pnl_events
        ),
    )
)

### Derive drawdown-state features

In [58]:
trade_features_table[
    "realized_return_before_trade"
] = (
    trade_features_table[
        "realized_pnl_before_trade"
    ]
    / STARTING_BALANCE
)

trade_features_table[
    "realized_balance_before_trade"
] = (
    STARTING_BALANCE
    + trade_features_table[
        "realized_pnl_before_trade"
    ]
)

trade_features_table[
    "realized_distance_to_drawdown_limit"
] = (
    trade_features_table[
        "realized_pnl_before_trade"
    ]
    - REALIZED_DRAWDOWN_BOUNDARY
)

trade_features_table[
    "realized_distance_to_profit_target"
] = (
    PROFIT_TARGET_AMOUNT
    - trade_features_table[
        "realized_pnl_before_trade"
    ]
)

## Data dictionary

In [59]:
feature_dictionary_records.extend(
    [
        {
            "feature_family": (
                "Challenge State and Drawdown Pressure"
            ),
            "feature_name": (
                "realized_pnl_before_trade"
            ),
            "level": "trade",
            "meaning": (
                "Cumulative realized P&L known before the current entry."
            ),
            "behaviour": (
                "Current challenge performance state."
            ),
            "calculation": (
                "Cumulative net profit from trades closed on or before "
                "the current opening timestamp."
            ),
            "pre_trade_only": True,
            "tested": False,
        },
        {
            "feature_family": (
                "Challenge State and Drawdown Pressure"
            ),
            "feature_name": (
                "realized_balance_before_trade"
            ),
            "level": "trade",
            "meaning": (
                "Realized account balance before the current trade."
            ),
            "behaviour": (
                "Account state relative to starting capital."
            ),
            "calculation": (
                "Starting balance plus realized_pnl_before_trade."
            ),
            "pre_trade_only": True,
            "tested": False,
        },
        {
            "feature_family": (
                "Challenge State and Drawdown Pressure"
            ),
            "feature_name": (
                "realized_distance_to_drawdown_limit"
            ),
            "level": "trade",
            "meaning": (
                "Remaining realized-P&L distance to the platform "
                "drawdown boundary."
            ),
            "behaviour": (
                "Drawdown pressure."
            ),
            "calculation": (
                "realized_pnl_before_trade minus the negative "
                "drawdown boundary."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": (
                "Challenge State and Drawdown Pressure"
            ),
            "feature_name": (
                "realized_distance_to_profit_target"
            ),
            "level": "trade",
            "meaning": (
                "Remaining realized profit required to reach "
                "the challenge target."
            ),
            "behaviour": (
                "Profit-target pressure."
            ),
            "calculation": (
                "Profit target amount minus realized_pnl_before_trade."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
    ]
)

## Test features

### Drawdown pressure

In [60]:
drawdown_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "realized_distance_to_drawdown_limit"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "realized_pnl_before_trade",
            "realized_distance_to_drawdown_limit",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

drawdown_results = (
    walk_forward_quantile_feature(
        data=drawdown_test_data,
        campaign_order=campaign_order,
        feature_column=(
            "realized_distance_to_drawdown_limit"
        ),
        signal_direction="low",
        candidate_quantiles=(
            LOW_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=drawdown_results,
        feature_family=(
            "Challenge State and Drawdown Pressure"
        ),
        feature_name=(
            "Drawdown pressure"
        ),
        feature_column=(
            "realized_distance_to_drawdown_limit"
        ),
        signal_description=(
            "Realized account state is unusually close to the "
            "drawdown boundary."
        ),
    )
)

### Profit-target proximity

In [61]:
profit_target_test_data = (
    trade_features_table.loc[
        (
            trade_features_table[
                "realized_pnl_before_trade"
            ] > 0
        )
        & (
            trade_features_table[
                "realized_pnl_before_trade"
            ]
            < PROFIT_TARGET_AMOUNT
        )
        & trade_features_table[
            "realized_distance_to_profit_target"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "realized_pnl_before_trade",
            "realized_distance_to_profit_target",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

profit_target_results = (
    walk_forward_quantile_feature(
        data=(
            profit_target_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "realized_distance_to_profit_target"
        ),
        signal_direction="low",
        candidate_quantiles=(
            LOW_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=profit_target_results,
        feature_family=(
            "Challenge State and Drawdown Pressure"
        ),
        feature_name=(
            "Profit-target proximity"
        ),
        feature_column=(
            "realized_distance_to_profit_target"
        ),
        signal_description=(
            "Account is unusually close to the realized "
            "profit target."
        ),
    )
)

# Position sizing

## Hypothesis

The predictiveness of position size may depend less on absolute lot size than
on whether the current size is unusual relative to the trader's own history,
or whether the trader has developed an inconsistent sizing pattern.

## Build features

### Historical median position size

In [62]:
def attach_historical_position_size_features(
    target_table: pd.DataFrame,
) -> pd.DataFrame:
    """Adds position-size features using strictly earlier trade entries.

    Trades opened at the same timestamp do not observe one another.

    Args:
        target_table: One-row-per-trade feature table.

    Returns:
        Table containing historical median size and current-to-history ratio.
    """
    result_frames = []

    for _, group in target_table.groupby(
        [
            "campaign_id",
            "account_id",
        ],
        sort=False,
        dropna=False,
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_medians = []
        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        for row in current.itertuples():
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )
                same_timestamp_amounts = []

            historical_median = (
                np.median(
                    historical_amounts
                )
                if historical_amounts
                else np.nan
            )

            historical_medians.append(
                historical_median
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "historical_median_amount_before_trade"
        ] = historical_medians

        result_frames.append(
            current
        )

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result[
        "current_to_historical_median_amount_ratio"
    ] = (
        result["amount"]
        / result[
            "historical_median_amount_before_trade"
        ]
    )

    result[
        "historical_size_available"
    ] = (
        result[
            "historical_median_amount_before_trade"
        ].gt(0)
    )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(drop=True)
    )

In [63]:
trade_features_table = (
    attach_historical_position_size_features(
        trade_features_table
    )
)

### Historical sizing variability

In [64]:
def attach_historical_sizing_consistency_features(
    target_table: pd.DataFrame,
) -> pd.DataFrame:
    """Adds pre-trade historical position-size variability features.

    Historical sizing uses only trades opened strictly before the current
    trade within the same account and campaign. Simultaneous entries do not
    observe one another.

    Args:
        target_table: One-row-per-trade feature table.

    Returns:
        Table containing historical sizing count, mean, standard deviation,
        and coefficient of variation.
    """
    result_frames = []

    for _, group in target_table.groupby(
        [
            "campaign_id",
            "account_id",
        ],
        sort=False,
        dropna=False,
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        historical_amounts = []
        same_timestamp_amounts = []
        previous_open_time = None

        past_counts = []
        past_means = []
        past_stds = []
        past_cvs = []

        for row in current.itertuples():
            current_open_time = (
                row.open_date_time
            )

            if (
                previous_open_time
                is not None
                and current_open_time
                != previous_open_time
            ):
                historical_amounts.extend(
                    same_timestamp_amounts
                )
                same_timestamp_amounts = []

            past_count = len(
                historical_amounts
            )

            past_mean = (
                np.mean(
                    historical_amounts
                )
                if past_count >= 1
                else np.nan
            )

            past_std = (
                np.std(
                    historical_amounts,
                    ddof=0,
                )
                if past_count >= 2
                else np.nan
            )

            past_cv = (
                past_std / past_mean
                if (
                    past_count >= 2
                    and past_mean > 0
                )
                else np.nan
            )

            past_counts.append(
                past_count
            )
            past_means.append(
                past_mean
            )
            past_stds.append(
                past_std
            )
            past_cvs.append(
                past_cv
            )

            same_timestamp_amounts.append(
                row.amount
            )

            previous_open_time = (
                current_open_time
            )

        current[
            "past_amount_count"
        ] = past_counts

        current[
            "past_amount_mean"
        ] = past_means

        current[
            "past_amount_std"
        ] = past_stds

        current[
            "past_amount_cv"
        ] = past_cvs

        result_frames.append(
            current
        )

    return (
        pd.concat(
            result_frames,
            ignore_index=True,
        )
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(drop=True)
    )

In [65]:
trade_features_table = (
    attach_historical_sizing_consistency_features(
        trade_features_table
    )
)

## Data dictionary

In [66]:
feature_dictionary_records.extend(
    [
        {
            "feature_family": "Position Sizing",
            "feature_name": (
                "historical_median_amount_before_trade"
            ),
            "level": "trade",
            "meaning": (
                "Median position size from strictly earlier trades."
            ),
            "behaviour": (
                "Trader-specific historical sizing baseline."
            ),
            "calculation": (
                "Median amount across earlier trade entries within "
                "the same account and campaign."
            ),
            "pre_trade_only": True,
            "tested": False,
        },
        {
            "feature_family": "Position Sizing",
            "feature_name": (
                "current_to_historical_median_amount_ratio"
            ),
            "level": "trade",
            "meaning": (
                "Current trade size relative to the trader's historical "
                "median size."
            ),
            "behaviour": (
                "Unusually large current position sizing."
            ),
            "calculation": (
                "Current amount divided by historical median amount."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": "Position Sizing",
            "feature_name": "past_amount_cv",
            "level": "trade",
            "meaning": (
                "Coefficient of variation of historical position sizes."
            ),
            "behaviour": (
                "Historical sizing consistency or instability."
            ),
            "calculation": (
                "Historical amount standard deviation divided by "
                "historical mean amount, using strictly earlier trades."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
    ]
)

## Test features

### Large current size

In [67]:
historical_size_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "historical_size_available"
        ]
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "current_to_historical_median_amount_ratio",
            "realized_pnl_before_trade",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

large_size_results = (
    walk_forward_quantile_feature(
        data=historical_size_test_data,
        campaign_order=campaign_order,
        feature_column=(
            "current_to_historical_median_amount_ratio"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=large_size_results,
        feature_family="Position Sizing",
        feature_name=(
            "Large current size relative to history"
        ),
        feature_column=(
            "current_to_historical_median_amount_ratio"
        ),
        signal_description=(
            "Current trade size is unusually large relative "
            "to the trader's historical median size."
        ),
    )
)

### Large size while in drawdown

In [68]:
drawdown_size_test_data = (
    historical_size_test_data.loc[
        historical_size_test_data[
            "realized_pnl_before_trade"
        ] < 0
    ]
    .copy()
)

large_size_drawdown_results = (
    walk_forward_quantile_feature(
        data=drawdown_size_test_data,
        campaign_order=campaign_order,
        feature_column=(
            "current_to_historical_median_amount_ratio"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=(
            large_size_drawdown_results
        ),
        feature_family="Position Sizing",
        feature_name=(
            "Large size during drawdown"
        ),
        feature_column=(
            "current_to_historical_median_amount_ratio"
        ),
        signal_description=(
            "Current size is unusually large while realized "
            "P&L is below starting balance."
        ),
    )
)

### Historical sizing variability

In [69]:
sizing_variability_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_amount_count"
        ].ge(3)
        & trade_features_table[
            "past_amount_cv"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_amount_count",
            "past_amount_cv",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

sizing_variability_results = (
    walk_forward_quantile_feature(
        data=(
            sizing_variability_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "past_amount_cv"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=(
            sizing_variability_results
        ),
        feature_family=(
            "Position Sizing"
        ),
        feature_name=(
            "High historical sizing variability"
        ),
        feature_column=(
            "past_amount_cv"
        ),
        signal_description=(
            "Historical position-size coefficient of variation "
            "is unusually high."
        ),
    )
)

# Stop-loss and take-profit behavior

## Hypothesis

The trader's planned risk-management configuration at entry may reveal
behavioural discipline or asymmetric risk taking. Features measure whether
stop-loss and take-profit levels are present and their distances from entry.

## Build features

In [70]:
trade_features_table[
    "has_stop_loss"
] = (
    trade_features_table[
        "sl_price"
    ].notna()
)

trade_features_table[
    "has_take_profit"
] = (
    trade_features_table[
        "tp_price"
    ].notna()
)

trade_features_table[
    "has_both_sl_tp"
] = (
    trade_features_table[
        "has_stop_loss"
    ]
    & trade_features_table[
        "has_take_profit"
    ]
)

trade_features_table[
    "has_neither_sl_tp"
] = (
    ~trade_features_table[
        "has_stop_loss"
    ]
    & ~trade_features_table[
        "has_take_profit"
    ]
)

### SL / TP configuration

In [71]:
trade_features_table[
    "sl_tp_configuration"
] = np.select(
    [
        (
            trade_features_table[
                "has_stop_loss"
            ]
            & trade_features_table[
                "has_take_profit"
            ]
        ),
        (
            trade_features_table[
                "has_stop_loss"
            ]
            & ~trade_features_table[
                "has_take_profit"
            ]
        ),
        (
            ~trade_features_table[
                "has_stop_loss"
            ]
            & trade_features_table[
                "has_take_profit"
            ]
        ),
        (
            ~trade_features_table[
                "has_stop_loss"
            ]
            & ~trade_features_table[
                "has_take_profit"
            ]
        ),
    ],
    [
        "both_sl_tp",
        "sl_only",
        "tp_only",
        "neither_sl_tp",
    ],
    default="unknown",
)

### Directional distances

In [72]:
buy_mask = (
    trade_features_table[
        "side"
    ].str.lower().eq("buy")
)

sell_mask = (
    trade_features_table[
        "side"
    ].str.lower().eq("sell")
)

trade_features_table[
    "stop_loss_distance"
] = np.nan

trade_features_table[
    "take_profit_distance"
] = np.nan

In [73]:
trade_features_table.loc[
    buy_mask
    & trade_features_table[
        "has_stop_loss"
    ],
    "stop_loss_distance",
] = (
    trade_features_table.loc[
        buy_mask
        & trade_features_table[
            "has_stop_loss"
        ],
        "open_price",
    ]
    - trade_features_table.loc[
        buy_mask
        & trade_features_table[
            "has_stop_loss"
        ],
        "sl_price",
    ]
)

trade_features_table.loc[
    sell_mask
    & trade_features_table[
        "has_stop_loss"
    ],
    "stop_loss_distance",
] = (
    trade_features_table.loc[
        sell_mask
        & trade_features_table[
            "has_stop_loss"
        ],
        "sl_price",
    ]
    - trade_features_table.loc[
        sell_mask
        & trade_features_table[
            "has_stop_loss"
        ],
        "open_price",
    ]
)

In [74]:
trade_features_table.loc[
    buy_mask
    & trade_features_table[
        "has_take_profit"
    ],
    "take_profit_distance",
] = (
    trade_features_table.loc[
        buy_mask
        & trade_features_table[
            "has_take_profit"
        ],
        "tp_price",
    ]
    - trade_features_table.loc[
        buy_mask
        & trade_features_table[
            "has_take_profit"
        ],
        "open_price",
    ]
)

trade_features_table.loc[
    sell_mask
    & trade_features_table[
        "has_take_profit"
    ],
    "take_profit_distance",
] = (
    trade_features_table.loc[
        sell_mask
        & trade_features_table[
            "has_take_profit"
        ],
        "open_price",
    ]
    - trade_features_table.loc[
        sell_mask
        & trade_features_table[
            "has_take_profit"
        ],
        "tp_price",
    ]
)

### Valid positive distances and normalized distances

In [75]:
trade_features_table[
    "valid_stop_loss_distance"
] = (
    trade_features_table[
        "stop_loss_distance"
    ]
    .where(
        trade_features_table[
            "stop_loss_distance"
        ] > 0
    )
)

trade_features_table[
    "valid_take_profit_distance"
] = (
    trade_features_table[
        "take_profit_distance"
    ]
    .where(
        trade_features_table[
            "take_profit_distance"
        ] > 0
    )
)

trade_features_table[
    "stop_loss_distance_pct"
] = (
    trade_features_table[
        "valid_stop_loss_distance"
    ]
    / trade_features_table[
        "open_price"
    ]
)

trade_features_table[
    "take_profit_distance_pct"
] = (
    trade_features_table[
        "valid_take_profit_distance"
    ]
    / trade_features_table[
        "open_price"
    ]
)

trade_features_table[
    "planned_reward_to_risk_ratio"
] = (
    trade_features_table[
        "valid_take_profit_distance"
    ]
    / trade_features_table[
        "valid_stop_loss_distance"
    ]
)

## Data dictionary

In [76]:
feature_dictionary_records.extend(
    [
        {
            "feature_family": (
                "Stop-Loss and Take-Profit Behaviour"
            ),
            "feature_name": "has_stop_loss",
            "level": "trade",
            "meaning": "Whether a stop-loss price is recorded.",
            "behaviour": "Stop-loss use.",
            "calculation": "sl_price is non-missing.",
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": (
                "Stop-Loss and Take-Profit Behaviour"
            ),
            "feature_name": "has_take_profit",
            "level": "trade",
            "meaning": "Whether a take-profit price is recorded.",
            "behaviour": "Take-profit use.",
            "calculation": "tp_price is non-missing.",
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": (
                "Stop-Loss and Take-Profit Behaviour"
            ),
            "feature_name": "sl_tp_configuration",
            "level": "trade",
            "meaning": (
                "Combined SL/TP configuration: both, SL only, "
                "TP only, or neither."
            ),
            "behaviour": "Planned risk-management configuration.",
            "calculation": (
                "Categorical combination of has_stop_loss "
                "and has_take_profit."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": (
                "Stop-Loss and Take-Profit Behaviour"
            ),
            "feature_name": "stop_loss_distance_pct",
            "level": "trade",
            "meaning": (
                "Direction-adjusted stop-loss distance relative "
                "to entry price."
            ),
            "behaviour": "Planned downside tolerance.",
            "calculation": (
                "Positive directional SL distance divided by open price."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": (
                "Stop-Loss and Take-Profit Behaviour"
            ),
            "feature_name": "take_profit_distance_pct",
            "level": "trade",
            "meaning": (
                "Direction-adjusted take-profit distance relative "
                "to entry price."
            ),
            "behaviour": "Planned upside target distance.",
            "calculation": (
                "Positive directional TP distance divided by open price."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": (
                "Stop-Loss and Take-Profit Behaviour"
            ),
            "feature_name": (
                "planned_reward_to_risk_ratio"
            ),
            "level": "trade",
            "meaning": (
                "Planned take-profit distance relative to "
                "stop-loss distance."
            ),
            "behaviour": "Planned reward-to-risk structure.",
            "calculation": (
                "Valid TP distance divided by valid SL distance."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
    ]
)

## Test features

### No stop loss

In [77]:
sl_tp_binary_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "has_stop_loss",
            "has_take_profit",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

no_stop_loss_results = (
    walk_forward_binary_feature(
        data=sl_tp_binary_test_data,
        campaign_order=campaign_order,
        signal_column="has_stop_loss",
        signal_value=False,
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=no_stop_loss_results,
        feature_family=(
            "Stop-Loss and Take-Profit Behaviour"
        ),
        feature_name="No stop loss",
        feature_column="has_stop_loss",
        signal_description=(
            "No stop-loss price is recorded for the trade."
        ),
    )
)

### No take profit

In [78]:
no_take_profit_results = (
    walk_forward_binary_feature(
        data=sl_tp_binary_test_data,
        campaign_order=campaign_order,
        signal_column="has_take_profit",
        signal_value=False,
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=no_take_profit_results,
        feature_family=(
            "Stop-Loss and Take-Profit Behaviour"
        ),
        feature_name="No take profit",
        feature_column="has_take_profit",
        signal_description=(
            "No take-profit price is recorded for the trade."
        ),
    )
)

### Configuration comparisons

In [79]:
sl_tp_configuration_data = (
    trade_features_table.loc[
        trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "sl_tp_configuration",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

In [80]:
for signal_value, feature_name in [
    (
        "sl_only",
        "SL only vs both SL and TP",
    ),
    (
        "tp_only",
        "TP only vs both SL and TP",
    ),
    (
        "neither_sl_tp",
        "Neither SL nor TP vs both",
    ),
]:
    configuration_results = (
        walk_forward_categorical_feature(
            data=(
                sl_tp_configuration_data
            ),
            campaign_order=(
                campaign_order
            ),
            feature_column=(
                "sl_tp_configuration"
            ),
            signal_value=(
                signal_value
            ),
            reference_value=(
                "both_sl_tp"
            ),
        )
    )

    feature_check_records.append(
        summarize_walk_forward(
            results=(
                configuration_results
            ),
            feature_family=(
                "Stop-Loss and Take-Profit Behaviour"
            ),
            feature_name=(
                feature_name
            ),
            feature_column=(
                "sl_tp_configuration"
            ),
            signal_description=(
                f"{signal_value} trades compared with "
                "trades containing both SL and TP."
            ),
        )
    )

### Wide take-profit distance

In [81]:
take_profit_distance_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "take_profit_distance_pct"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "take_profit_distance_pct",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

wide_tp_results = (
    walk_forward_quantile_feature(
        data=(
            take_profit_distance_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "take_profit_distance_pct"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=wide_tp_results,
        feature_family=(
            "Stop-Loss and Take-Profit Behaviour"
        ),
        feature_name=(
            "Wide take-profit distance"
        ),
        feature_column=(
            "take_profit_distance_pct"
        ),
        signal_description=(
            "Take-profit distance is unusually wide "
            "relative to entry price."
        ),
    )
)

### Wide stop-loss distance

In [82]:
stop_loss_distance_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "stop_loss_distance_pct"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "stop_loss_distance_pct",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

wide_sl_results = (
    walk_forward_quantile_feature(
        data=(
            stop_loss_distance_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "stop_loss_distance_pct"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=wide_sl_results,
        feature_family=(
            "Stop-Loss and Take-Profit Behaviour"
        ),
        feature_name=(
            "Wide stop-loss distance"
        ),
        feature_column=(
            "stop_loss_distance_pct"
        ),
        signal_description=(
            "Stop-loss distance is unusually wide "
            "relative to entry price."
        ),
    )
)

### High reward-to-risk ratio

In [83]:
reward_to_risk_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "planned_reward_to_risk_ratio"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "planned_reward_to_risk_ratio",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

reward_to_risk_results = (
    walk_forward_quantile_feature(
        data=(
            reward_to_risk_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "planned_reward_to_risk_ratio"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=(
            reward_to_risk_results
        ),
        feature_family=(
            "Stop-Loss and Take-Profit Behaviour"
        ),
        feature_name=(
            "High planned reward-to-risk ratio"
        ),
        feature_column=(
            "planned_reward_to_risk_ratio"
        ),
        signal_description=(
            "Planned reward-to-risk ratio is unusually high."
        ),
    )
)

# Historical performance

## Hypothesis

A trader's completed idea history may contain information about the quality of
their subsequent trades. Historical win rate, payoff ratio, and loss magnitude
are constructed using only ideas completed before the current idea begins.

## Build features

In [84]:
def build_historical_performance_features(
    idea_table: pd.DataFrame,
) -> pd.DataFrame:
    """Builds pre-idea historical performance features.

    Historical information comes only from ideas whose end timestamp is
    strictly earlier than the current idea start. History is accumulated
    by account across completed ideas.

    Args:
        idea_table: One-row-per-idea table containing idea timing, size,
            profitability, and outcome indicators.

    Returns:
        One-row-per-idea table containing historical win rate, average win,
        average loss magnitude, payoff ratio, and historical counts.
    """
    idea_performance = (
        idea_table[
            [
                "idea_id",
                "account_id",
                "campaign_id",
                "idea_start_time",
                "idea_end_time",
                "total_amount",
                "total_net_profit",
                "is_profitable_idea",
                "is_losing_idea",
                "is_breakeven_idea",
            ]
        ]
        .copy()
    )

    idea_performance[
        "idea_profit_per_lot"
    ] = (
        idea_performance[
            "total_net_profit"
        ]
        / idea_performance[
            "total_amount"
        ]
    )

    idea_performance[
        "_win_count"
    ] = (
        idea_performance[
            "is_profitable_idea"
        ].astype(int)
    )

    idea_performance[
        "_loss_count"
    ] = (
        idea_performance[
            "is_losing_idea"
        ].astype(int)
    )

    idea_performance[
        "_idea_count"
    ] = 1

    idea_performance[
        "_win_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .where(
            idea_performance[
                "is_profitable_idea"
            ],
            0.0,
        )
    )

    idea_performance[
        "_loss_abs_profit"
    ] = (
        idea_performance[
            "idea_profit_per_lot"
        ]
        .abs()
        .where(
            idea_performance[
                "is_losing_idea"
            ],
            0.0,
        )
    )

    events = (
        idea_performance
        .groupby(
            [
                "account_id",
                "idea_end_time",
            ],
            as_index=False,
        )
        .agg(
            completed_idea_count=(
                "_idea_count",
                "sum",
            ),
            completed_win_count=(
                "_win_count",
                "sum",
            ),
            completed_loss_count=(
                "_loss_count",
                "sum",
            ),
            completed_win_profit_sum=(
                "_win_profit",
                "sum",
            ),
            completed_loss_abs_profit_sum=(
                "_loss_abs_profit",
                "sum",
            ),
        )
        .sort_values(
            [
                "account_id",
                "idea_end_time",
            ]
        )
        .reset_index(drop=True)
    )

    cumulative_mapping = {
        "past_completed_idea_count": (
            "completed_idea_count"
        ),
        "past_winning_idea_count": (
            "completed_win_count"
        ),
        "past_losing_idea_count": (
            "completed_loss_count"
        ),
        "past_win_profit_sum": (
            "completed_win_profit_sum"
        ),
        "past_loss_abs_profit_sum": (
            "completed_loss_abs_profit_sum"
        ),
    }

    for output_column, source_column in (
        cumulative_mapping.items()
    ):
        events[
            output_column
        ] = (
            events
            .groupby(
                "account_id"
            )[source_column]
            .cumsum()
        )

    current_ideas = (
        idea_performance[
            [
                "idea_id",
                "account_id",
                "campaign_id",
                "idea_start_time",
            ]
        ]
        .sort_values(
            [
                "idea_start_time",
                "account_id",
            ]
        )
        .reset_index(drop=True)
    )

    events = (
        events
        .sort_values(
            [
                "idea_end_time",
                "account_id",
            ]
        )
        .reset_index(drop=True)
    )

    historical = pd.merge_asof(
        current_ideas,
        events[
            [
                "account_id",
                "idea_end_time",
                "past_completed_idea_count",
                "past_winning_idea_count",
                "past_losing_idea_count",
                "past_win_profit_sum",
                "past_loss_abs_profit_sum",
            ]
        ],
        left_on="idea_start_time",
        right_on="idea_end_time",
        by="account_id",
        direction="backward",
        allow_exact_matches=False,
    )

    historical[
        "past_idea_win_rate"
    ] = (
        historical[
            "past_winning_idea_count"
        ]
        / historical[
            "past_completed_idea_count"
        ]
    )

    historical[
        "past_mean_win_profit_per_lot"
    ] = (
        historical[
            "past_win_profit_sum"
        ]
        / historical[
            "past_winning_idea_count"
        ]
    )

    historical[
        "past_mean_loss_abs_profit_per_lot"
    ] = (
        historical[
            "past_loss_abs_profit_sum"
        ]
        / historical[
            "past_losing_idea_count"
        ]
    )

    historical[
        "past_payoff_ratio"
    ] = (
        historical[
            "past_mean_win_profit_per_lot"
        ]
        / historical[
            "past_mean_loss_abs_profit_per_lot"
        ]
    )

    return historical

In [85]:
historical_idea_features = (
    build_historical_performance_features(
        idea_features_table
    )
)

In [86]:
historical_performance_columns = [
    "idea_id",
    "past_completed_idea_count",
    "past_winning_idea_count",
    "past_losing_idea_count",
    "past_idea_win_rate",
    "past_mean_win_profit_per_lot",
    "past_mean_loss_abs_profit_per_lot",
    "past_payoff_ratio",
]

idea_features_table = (
    idea_features_table
    .merge(
        historical_idea_features[
            historical_performance_columns
        ],
        on="idea_id",
        how="left",
        validate="one_to_one",
    )
)

In [87]:
trade_features_table = (
    trade_features_table
    .merge(
        historical_idea_features[
            historical_performance_columns
        ],
        on="idea_id",
        how="left",
        validate="many_to_one",
    )
)

## Data dictionary

In [88]:
feature_dictionary_records.extend(
    [
        {
            "feature_family": "Historical Performance",
            "feature_name": "past_idea_win_rate",
            "level": "idea / trade",
            "meaning": (
                "Share of previously completed ideas that were profitable."
            ),
            "behaviour": (
                "Historical hit-rate performance."
            ),
            "calculation": (
                "Past winning ideas divided by all past completed ideas."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": "Historical Performance",
            "feature_name": (
                "past_mean_win_profit_per_lot"
            ),
            "level": "idea / trade",
            "meaning": (
                "Average profit per lot among historical winning ideas."
            ),
            "behaviour": "Historical upside magnitude.",
            "calculation": (
                "Sum of historical winning idea profit-per-lot "
                "divided by historical winning idea count."
            ),
            "pre_trade_only": True,
            "tested": False,
        },
        {
            "feature_family": "Historical Performance",
            "feature_name": (
                "past_mean_loss_abs_profit_per_lot"
            ),
            "level": "idea / trade",
            "meaning": (
                "Average absolute loss per lot among historical "
                "losing ideas."
            ),
            "behaviour": "Historical downside magnitude.",
            "calculation": (
                "Sum of absolute historical losing idea "
                "profit-per-lot divided by losing idea count."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
        {
            "feature_family": "Historical Performance",
            "feature_name": "past_payoff_ratio",
            "level": "idea / trade",
            "meaning": (
                "Historical average win magnitude relative to "
                "average loss magnitude."
            ),
            "behaviour": "Historical payoff quality.",
            "calculation": (
                "past_mean_win_profit_per_lot divided by "
                "past_mean_loss_abs_profit_per_lot."
            ),
            "pre_trade_only": True,
            "tested": True,
        },
    ]
)

## Test features

### Low historical win rate

In [89]:
historical_win_rate_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_idea_win_rate"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_idea_win_rate",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

historical_win_rate_results = (
    walk_forward_quantile_feature(
        data=(
            historical_win_rate_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "past_idea_win_rate"
        ),
        signal_direction="low",
        candidate_quantiles=(
            LOW_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=(
            historical_win_rate_results
        ),
        feature_family=(
            "Historical Performance"
        ),
        feature_name=(
            "Low historical win rate"
        ),
        feature_column=(
            "past_idea_win_rate"
        ),
        signal_description=(
            "Historical completed-idea win rate is unusually low."
        ),
    )
)

### Low historical payoff ratio

In [90]:
historical_payoff_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_payoff_ratio"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_payoff_ratio",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

historical_payoff_results = (
    walk_forward_quantile_feature(
        data=(
            historical_payoff_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "past_payoff_ratio"
        ),
        signal_direction="low",
        candidate_quantiles=(
            LOW_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=(
            historical_payoff_results
        ),
        feature_family=(
            "Historical Performance"
        ),
        feature_name=(
            "Low historical payoff ratio"
        ),
        feature_column=(
            "past_payoff_ratio"
        ),
        signal_description=(
            "Historical average win-to-loss magnitude ratio "
            "is unusually low."
        ),
    )
)

### High historical loss magnitude

In [91]:
historical_loss_test_data = (
    trade_features_table.loc[
        trade_features_table[
            "past_mean_loss_abs_profit_per_lot"
        ].notna()
        & trade_features_table[
            "reverse_profit_per_lot"
        ].notna(),
        [
            "trade_row_id",
            "account_id",
            "campaign_id",
            "past_mean_loss_abs_profit_per_lot",
            "reverse_profit_per_lot",
        ],
    ]
    .copy()
)

historical_loss_results = (
    walk_forward_quantile_feature(
        data=(
            historical_loss_test_data
        ),
        campaign_order=campaign_order,
        feature_column=(
            "past_mean_loss_abs_profit_per_lot"
        ),
        signal_direction="high",
        candidate_quantiles=(
            HIGH_SIGNAL_QUANTILES
        ),
    )
)

feature_check_records.append(
    summarize_walk_forward(
        results=historical_loss_results,
        feature_family=(
            "Historical Performance"
        ),
        feature_name=(
            "High historical loss magnitude"
        ),
        feature_column=(
            "past_mean_loss_abs_profit_per_lot"
        ),
        signal_description=(
            "Historical average loss magnitude per lot is "
            "unusually high."
        ),
    )
)

# Feature check summary

In [92]:
feature_check_results = (
    pd.DataFrame(
        feature_check_records
    )
    .sort_values(
        [
            "feature_family",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)

display(
    feature_check_results
)

,feature_family,feature_name,feature_column,signal_description,test_campaigns,mean_oos_uplift,median_oos_uplift,positive_campaign_share,ci_lower,ci_upper,robustness
0,Challenge State and Drawdown Pressure,Drawdown pressure,realized_distance_to_drawdown_limit,Realized account state is unusually close to t...,24,20.142014,28.406023,0.583333,3.312718,37.328668,robust_positive
1,Challenge State and Drawdown Pressure,Profit-target proximity,realized_distance_to_profit_target,Account is unusually close to the realized pro...,24,13.161127,-6.114866,0.416667,-19.371522,48.747271,not_robust
2,Historical Performance,High historical loss magnitude,past_mean_loss_abs_profit_per_lot,Historical average loss magnitude per lot is u...,24,6.632898,14.681693,0.583333,-15.502521,28.491183,not_robust
3,Historical Performance,Low historical payoff ratio,past_payoff_ratio,Historical average win-to-loss magnitude ratio...,24,5.217508,-9.553544,0.416667,-36.299654,48.857789,not_robust
4,Historical Performance,Low historical win rate,past_idea_win_rate,Historical completed-idea win rate is unusuall...,24,-2.440202,-5.356596,0.458333,-25.854465,20.541176,not_robust
5,Loss Response,Fast post-loss re-entry,post_loss_reentry_gap_minutes,Post-loss re-entry gap is unusually short rela...,24,-1.169356,10.427381,0.583333,-31.077002,26.653175,not_robust
6,Loss Response,Fast re-entry with size escalation,post_loss_reentry_gap_minutes + post_loss_amou...,Trade both re-enters unusually quickly after a...,24,12.700406,4.532832,0.500000,-42.360008,65.673774,not_robust
7,Loss Response,Post-loss size escalation,post_loss_amount_increased,Current position size is larger than the previ...,24,0.698649,18.493088,0.625000,-38.777299,36.474287,not_robust
8,Position Sizing,High historical sizing variability,past_amount_cv,Historical position-size coefficient of variat...,24,25.992703,16.273575,0.750000,6.468000,49.846109,robust_positive
9,Position Sizing,Large current size relative to history,current_to_historical_median_amount_ratio,Current trade size is unusually large relative...,24,-1.930571,2.994883,0.500000,-24.078807,18.878529,not_robust


In [93]:
feature_check_results[
    "stage3_candidate"
] = (
    feature_check_results[
        "robustness"
    ].isin(
        [
            "robust_positive",
            "robust_negative",
        ]
    )
)

# Data dictionary

In [94]:
feature_dictionary = (
    pd.DataFrame(
        feature_dictionary_records
    )
    .drop_duplicates(
        subset=[
            "feature_name",
            "level",
        ]
    )
    .sort_values(
        [
            "feature_family",
            "feature_name",
        ]
    )
    .reset_index(drop=True)
)

display(
    feature_dictionary
)

,feature_family,feature_name,level,meaning,behaviour,calculation,pre_trade_only,tested
0,Challenge State and Drawdown Pressure,realized_balance_before_trade,trade,Realized account balance before the current tr...,Account state relative to starting capital.,Starting balance plus realized_pnl_before_trade.,True,False
1,Challenge State and Drawdown Pressure,realized_distance_to_drawdown_limit,trade,Remaining realized-P&L distance to the platfor...,Drawdown pressure.,realized_pnl_before_trade minus the negative d...,True,True
2,Challenge State and Drawdown Pressure,realized_distance_to_profit_target,trade,Remaining realized profit required to reach th...,Profit-target pressure.,Profit target amount minus realized_pnl_before...,True,True
3,Challenge State and Drawdown Pressure,realized_pnl_before_trade,trade,Cumulative realized P&L known before the curre...,Current challenge performance state.,Cumulative net profit from trades closed on or...,True,False
4,Historical Performance,past_idea_win_rate,idea / trade,Share of previously completed ideas that were ...,Historical hit-rate performance.,Past winning ideas divided by all past complet...,True,True
5,Historical Performance,past_mean_loss_abs_profit_per_lot,idea / trade,Average absolute loss per lot among historical...,Historical downside magnitude.,Sum of absolute historical losing idea profit-...,True,True
6,Historical Performance,past_mean_win_profit_per_lot,idea / trade,Average profit per lot among historical winnin...,Historical upside magnitude.,Sum of historical winning idea profit-per-lot ...,True,False
7,Historical Performance,past_payoff_ratio,idea / trade,Historical average win magnitude relative to a...,Historical payoff quality.,past_mean_win_profit_per_lot divided by past_m...,True,True
8,Loss Response,post_loss_amount_increased,trade,Indicates that position size increased after t...,Captures post-loss size escalation.,post_loss_amount_ratio > 1 for a post-loss entry.,True,True
9,Loss Response,post_loss_amount_ratio,trade,Current trade amount relative to the amount of...,Measures position-size reaction after a loss.,Current amount divided by previous completed t...,True,True


# Finalize feature tables

In [95]:
trade_features_table = (
    trade_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
            "trade_row_id",
        ]
    )
    .reset_index(drop=True)
)

idea_features_table = (
    idea_features_table
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "idea_start_time",
            "idea_end_time",
            "idea_id",
        ]
    )
    .reset_index(drop=True)
)

# Export stage 2 outputs

In [96]:
STAGE2_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

In [97]:
trade_features_table.to_parquet(
    STAGE2_OUTPUT_DIR
    / "trade_features.parquet",
    index=False,
)

idea_features_table.to_parquet(
    STAGE2_OUTPUT_DIR
    / "idea_features.parquet",
    index=False,
)

feature_check_results.to_csv(
    STAGE2_OUTPUT_DIR
    / "feature_check_results.csv",
    index=False,
)

feature_dictionary.to_csv(
    STAGE2_OUTPUT_DIR
    / "feature_dictionary.csv",
    index=False,
)

In [98]:
print(
    "Stage 2 outputs saved to:",
    STAGE2_OUTPUT_DIR.resolve(),
)

print(
    "Trade-level feature table:",
    trade_features_table.shape,
)

print(
    "Idea-level feature table:",
    idea_features_table.shape,
)

print(
    "Documented features:",
    len(feature_dictionary),
)

print(
    "Feature checks:",
    len(feature_check_results),
)

Stage 2 outputs saved to: C:\Desktop\C22-veNTUre\outputs\stage2
Trade-level feature table: (46520, 132)
Idea-level feature table: (33379, 44)
Documented features: 24
Feature checks: 22
